# Aurelius v3 — Code-domain Verifier Recursion Run (resumable)

Runs the **VerifierRecursionLoop** where the verifier is ground truth: base -> generate k -> **execution-verify** -> select-correct -> RLVR fold-in -> repeat. This is the honest home for the lever (the math learned-verifier path was a measured null 2026-07-07; on code the execution verifier selects perfectly).

**Two questions:**
1. **Selection (free, §2):** how big is the code selection gap (greedy -> best-of-k = oracle)?
2. **Fold-in (RLVR, §3):** does training on selected-correct raise **greedy** pass@1 across cycles, or hit the winners-only regression (the arc's -11pp trap)?

Imports the tested `src/eval/vgbs.py` + `src/training/verifier_recursion.py` (60 tests) from the pushed `wip/alignment-evallab` branch. **Selection-first:** `TRAIN=False` (default) runs the offline-tested selection logic on real MBPP to prove the plumbing; flip `TRAIN=True` for the RLVR fold-in. Resumable per-cycle checkpoints to Drive. A100/L4-class GPU.

In [ ]:
# ===================== CONFIG =====================
MODEL = 'Qwen/Qwen3-4B'          # policy (set larger for the real run; ref copy only if TRAIN)
TRAIN = False                    # False = selection-only dry-run (tested logic); True = RLVR fold-in
N_CYCLES = 3
K = 8                            # candidates/task
N_TRAIN = 60                     # MBPP tasks used for recursion cycles
N_EVAL = 40                      # DISJOINT held-out MBPP for greedy pass@1 (the metric that must rise)
MAX_NEW = 512
LR = 1e-5                        # RLVR lr (NOT the inert 1e-6 from the withdrawn run)
WORK = '/content/aurelius_code_recursion'
CKPT_DIR = '/content/drive/MyDrive/aurelius_code_recursion'   # resumable per-cycle
BRANCH = 'wip/alignment-evallab'
REPO_URL, REPO_DIR = 'https://github.com/S3nna13/Aurelius', '/content/Aurelius'
import os; os.makedirs(WORK, exist_ok=True)
print('config | model', MODEL, '| TRAIN', TRAIN, '| cycles', N_CYCLES, 'k', K)

In [ ]:
# ===================== deps + clone branch (tested modules) + imports =====================
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.44','datasets','peft','accelerate'], check=False)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], check=False)  # breaks peft LoRA dispatch
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO_URL,REPO_DIR], check=False)
sys.path.insert(0, REPO_DIR)
from src.eval.vgbs import verified_best_of_n, cost_ratio
from src.training.verifier_recursion import (VerifierRecursionLoop, Task, recursion_verdict, make_code_reward)
print('imported tested lever modules | VGBS cost ratio', round(cost_ratio(),2), 'x')

## §0 — GPU preflight + resumable checkpoint dir

In [ ]:
import torch, json
assert torch.cuda.is_available(), 'no GPU'
dev='cuda'; vram=torch.cuda.get_device_properties(0).total_memory/1e9
print(torch.cuda.get_device_name(0), f'{vram:.0f}GB | TRAIN={TRAIN} (needs policy+ref ~2x if True)')
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs(CKPT_DIR, exist_ok=True); CKPT=CKPT_DIR
except Exception:
    CKPT=WORK; print('no Drive -> local checkpoints at', WORK)
def ck_path(name): return os.path.join(CKPT, name)
def save_ck(name, obj):
    p=ck_path(name); tmp=p+'.tmp'; json.dump(obj, open(tmp,'w')); os.replace(tmp,p)
def load_ck(name, default=None):
    p=ck_path(name); return json.load(open(p)) if os.path.exists(p) else default
print('checkpoints ->', CKPT)

## §1 — Load model + MBPP + execution test-runner (DISJOINT train/eval split)

In [ ]:
import re, tempfile, subprocess as sp
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, trust_remote_code=True).to(dev).eval()

mbpp = load_dataset('google-research-datasets/mbpp','full',split='test')
train_rows = mbpp.select(range(N_TRAIN))
eval_rows  = mbpp.select(range(N_TRAIN, N_TRAIN+N_EVAL))   # DISJOINT held-out for greedy
def mk_prompt(text): return text + '\n\nReturn only a Python function in a ```python block.'
def extract_code(t):
    m=re.search(r'```(?:python)?\n(.*?)```', t, re.DOTALL); return m.group(1) if m else t
def task_tests(ex):
    # MBPP problems often need test_setup_code (imports/helpers) or the asserts fail
    setup = [ex['test_setup_code']] if ex.get('test_setup_code') else []
    return setup + list(ex['test_list'])

def run_tests(completion, tests, timeout=8):
    prog = completion + '\n' + '\n'.join(tests)
    try:
        with tempfile.NamedTemporaryFile('w',suffix='.py',delete=False) as f: f.write(prog); p=f.name
        r=sp.run([sys.executable,p], capture_output=True, timeout=timeout); os.unlink(p)
        return (1,1) if r.returncode==0 else (0,1)
    except Exception: return (0,1)

def to_ids(prompt):
    # robust across transformers versions: apply_chat_template may return a bare
    # tensor OR a BatchEncoding (dict); some models reject enable_thinking.
    msgs=[{'role':'user','content':prompt}]
    try: enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', enable_thinking=False)
    except TypeError: enc=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt')
    return (enc if torch.is_tensor(enc) else enc['input_ids']).to(dev)

@torch.no_grad()
def gen_k(prompt, k, greedy=False):
    ids=to_ids(prompt)
    kw=dict(max_new_tokens=MAX_NEW, pad_token_id=tok.pad_token_id)
    if greedy: out=model.generate(ids, do_sample=False, **kw)
    else: out=model.generate(ids, do_sample=True, temperature=0.8, top_p=0.95, num_return_sequences=k, **kw)
    return [extract_code(tok.decode(o[ids.shape[1]:], skip_special_tokens=True)) for o in out]
print(f'loaded | train {len(train_rows)} / held-out eval {len(eval_rows)} (disjoint)')

## §2 — Selection dry-run (offline-tested logic on real MBPP; always runs)

Measures the code selection gap and proves the loop's selection + precision path with an execution verifier. `verified_best_of_n` = the code/terminal form of VGBS. With execution verification, verifier-selected == oracle (best-of-k captures the full gap for free).

In [ ]:
# per-task execution verifier (the loop's 2-arg verify_fn), bound to that task's tests
def make_verify(tests):
    def v(prompt, completion):
        p, t = run_tests(completion, tests); return 1.0 if (t > 0 and p == t) else 0.0
    return v

def eval_greedy(rows):
    ok = 0
    for ex in rows:
        c = gen_k(mk_prompt(ex['text']), 1, greedy=True)[0]
        p, t = run_tests(c, task_tests(ex)); ok += (t > 0 and p == t)
    return ok / len(rows)

# selection gap on the train set: REAL greedy vs verified best-of-k (== oracle for an exec verifier)
g = s = o = 0
for ex in train_rows:
    prompt = mk_prompt(ex['text'])
    tests = task_tests(ex)
    greedy_c = gen_k(prompt, 1, greedy=True)[0]            # true greedy decode (not a sample)
    cands = gen_k(prompt, K)                                # k temperature samples
    verify = make_verify(tests)
    _, score, scores = verified_best_of_n(prompt, cands, verify)
    gp, gt = run_tests(greedy_c, tests)
    g += (gt > 0 and gp == gt)                              # greedy pass@1
    s += score >= 1.0                                       # verified best-of-k
    o += any(x >= 1.0 for x in scores)                      # oracle@k
n = len(train_rows)
print(f'=== §2 selection on {n} MBPP (k={K}) ===')
print(f'  greedy        {g/n*100:.1f}%')
print(f'  verified BoN  {s/n*100:.1f}%   (== oracle for an execution verifier)')
print(f'  oracle@{K}      {o/n*100:.1f}%')
print(f'  selection gap +{(o-g)/n*100:.1f}pp  (greedy->oracle = what the fold-in targets)')
save_ck('selection.json', {'greedy': g/n, 'verified_bon': s/n, 'oracle': o/n, 'n': n})

## §3 — RLVR fold-in (TRAIN=True) — run cycles, measure GREEDY on held-out each cycle

Builds `CurriculumRLVRTrainer` with `make_code_reward` (the 3-arg RLVR reward — NOT the 2-arg selection verifier). Each cycle: generate k on train tasks -> execution-verify -> select-correct -> RLVR step; then re-measure **greedy pass@1 on the DISJOINT held-out set** (the metric that must rise). **LoRA on the policy** (full-param RLVR would OOM an 80GB box on 8B); frozen base copy = the KL ref. Resumable: skips cycles already checkpointed. **Correct-by-construction (RLVR loop can't be unit-tested offline); the §2 dry-run is the tested-logic gate — run it green first.**

In [ ]:
if not TRAIN:
    print('TRAIN=False -> selection-only dry-run done. Flip TRAIN=True for the RLVR fold-in.')
else:
    from src.alignment.rlvr import RLVRTrainer, RLVRConfig
    from src.training.grpo_trainer_with_curriculum import CurriculumRLVRTrainer
    from src.training.curriculum_rl import CurriculumRLSampler, CurriculumRLConfig
    from peft import LoraConfig, get_peft_model
    from torch.optim import AdamW

    # LoRA on the policy — full-param RLVR would OOM (8B Adam state ~128GB). Adapter
    # only -> tiny optimizer state; the frozen base copy is the KL reference.
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM'))
    model.print_trainable_parameters()
    ref = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, trust_remote_code=True).to(dev).eval()
    for p in ref.parameters(): p.requires_grad_(False)
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)  # LoRA params only
    # tests lookup (WITH test_setup_code): ground_truth = task id -> its full test block
    tests_by_id = {f't{i}': task_tests(ex) for i,ex in enumerate(train_rows)}
    mk_prompt_by_id = {f't{i}': mk_prompt(ex['text']) for i,ex in enumerate(train_rows)}
    prompt_to_id = {mk_prompt(ex['text']): f't{i}' for i,ex in enumerate(train_rows)}
    reward_fn = make_code_reward(run_tests, tests_for=lambda gt: tests_by_id[gt])
    sampler = CurriculumRLSampler(CurriculumRLConfig())
    for tid in tests_by_id: sampler.register_task(tid, difficulty=0.5)
    trainer = CurriculumRLVRTrainer(policy_model=model, ref_model=ref, optimizer=opt,
        reward_fn=reward_fn, rlvr_config=RLVRConfig(), sampler=sampler)

    def train_fn(selected):
        rew=[]
        for ex in selected:
            r = trainer.train_step(task_ids=[ex['task_id']], prompt_ids=to_ids(mk_prompt_by_id[ex['task_id']]),
                prompt_text=mk_prompt_by_id[ex['task_id']], answer=ex['task_id'])
            rew.append(float(r.get('mean_reward',0.0)))
        return {'mean_reward': sum(rew)/len(rew) if rew else 0.0}

    tasks = [Task(f't{i}', mk_prompt(ex['text']), answer=f't{i}') for i,ex in enumerate(train_rows)]
    def gen_for_loop(prompt, k): return gen_k(prompt, k)
    def verify_for_loop(prompt, completion):
        tid = prompt_to_id[prompt]; p,t = run_tests(completion, tests_by_id[tid]); return 1.0 if (t>0 and p==t) else 0.0
    loop = VerifierRecursionLoop(gen_for_loop, verify_for_loop, train_fn, k=K, select_threshold=1.0)

    hist = load_ck('cycles.json', default={'greedy_eval':[], 'cycles':[]})
    for cyc in range(len(hist['cycles']), N_CYCLES):
        gpre = eval_greedy(eval_rows)                       # greedy on held-out BEFORE this cycle's train
        res = loop.run_cycle(tasks, cycle=cyc)
        gpost = eval_greedy(eval_rows)                      # AFTER
        row = res.ledger_row(); row['greedy_eval_pre']=round(gpre,4); row['greedy_eval_post']=round(gpost,4)
        hist['cycles'].append(row); hist['greedy_eval'].append(gpost)
        save_ck('cycles.json', hist)
        print(f'cycle {cyc}: selected {res.n_selected}/{res.n_candidates} | greedy held-out {gpre*100:.1f}->{gpost*100:.1f}%')
    print('cycles done:', hist['greedy_eval'])

## §4 — Verdict + TruthSurface ledger row

In [ ]:
if TRAIN:
    hist = load_ck('cycles.json', default={'greedy_eval':[]})
    ge = hist['greedy_eval']
    if len(ge)>=2:
        from src.training.verifier_recursion import CycleResult
        cyc_objs=[CycleResult(cycle=i,n_tasks=N_TRAIN,n_candidates=0,n_selected=0,selected_frac=0,
                  solve_rate=g,verifier_precision=None,train_mean_reward=None,cost_generations=0) for i,g in enumerate(ge)]
        v = recursion_verdict(cyc_objs)
        print('=== §4 VERDICT (greedy held-out across cycles) ===')
        print(f'  greedy pass@1: {[round(x*100,1) for x in ge]}')
        print(f'  {v["verdict"]}  (d_solve {v["d_solve_rate"]:+.3f})')
        print('  COMPOUNDS = fold-in works (run more cycles). REGRESSES = winners-only trap. FLAT = ship best-of-N at inference.')
    else: print('need >=2 cycles for a verdict')
else:
    sel = load_ck('selection.json')
    print('selection dry-run:', sel)
    print('flip TRAIN=True to measure whether the fold-in raises greedy.')
# TruthSurface: append a claim row (proposed until >=2 cycles + significance)
try:
    from src.eval.truth_surface import TruthSurface, ClaimRecord
    ts=TruthSurface(ck_path('truthsurface.jsonl'))
    if TRAIN and len(load_ck('cycles.json',{}).get('greedy_eval',[]))>=2:
        ge=load_ck('cycles.json')['greedy_eval']
        ts.add(ClaimRecord('code-recursion-run','verifier-recursion folds code selection gap into greedy',
            metric='MBPP greedy pass@1 (held-out)', baseline=f'cycle-0 {ge[0]*100:.1f}%',
            falsifier='greedy falls (winners-only) or flat across cycles', rollback='ship best-of-N at inference',
            value=ge[-1]*100, baseline_value=ge[0]*100, n=N_EVAL, bench='mbpp'))
        print('TruthSurface row appended ->', ck_path('truthsurface.jsonl'))
except Exception as e: print('ledger note skipped:', e)

## Interpreting

- **§2 selection gap** = greedy -> oracle on code. Large gap = headroom the fold-in targets; with an execution verifier, best-of-k already captures it FREE at inference (ship that regardless).
- **§3/§4 fold-in verdict:** COMPOUNDS (greedy rises across cycles) = the recursion loop works on code where the verifier is perfect -> run more cycles. REGRESSES = the winners-only -11pp trap (don't ship the trained model; ship best-of-N). FLAT = the gap is only cashable at inference (best-of-N), not foldable into greedy.
- **Discipline:** run §2 GREEN first (tested selection logic). n is small -> DIRECTIONAL; feed results through `promotion_gate` before believing (a delta within CI half-width is a draw, not a win — the same gate that caught the RLVR +1.3pp mirage). Scale N_TRAIN/N_EVAL + cycles before any claim.